## Imports + configs

In [79]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

from lightgbm import LGBMClassifier

RANDOM_SEED = 42

# Toggle: run with all classes or drop some
DROP_CLASSES = True
DROP_WEAK_LINKS = {
    #"LeftHeelRises",
    "RightHeelRises",
    #"LeftKneeMovesInward",
    #"RightAsymmetricalWeightShift",
}

DROP_TIES = True

SCORES_PATH = "../../data/processed/20190108_scores_and_weak_links.csv"
FEATURES_PATH = "../../data/processed/AimoScore_WeakLink_big_scores.csv"


## Load and build Weakest link labels

In [80]:
scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

if DROP_CLASSES:
    indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

scores_labeled = scores.loc[~scores["HasTie"]].copy() if DROP_TIES else scores.copy()
scores_labeled = scores_labeled.dropna(subset=["WeakestLink"])

print("Unique WeakestLink classes:", scores_labeled["WeakestLink"].nunique())
display(scores_labeled["WeakestLink"].value_counts())


Unique WeakestLink classes: 13


WeakestLink
RightArmFallForward             452
LeftArmFallForward              322
RightKneeMovesOutward           275
RightShoulderElevation          245
ExcessiveForwardLean             94
ForwardHead                      92
LeftAsymmetricalWeightShift      71
LeftShoulderElevation            51
LeftKneeMovesOutward             47
RightKneeMovesInward             45
RightAsymmetricalWeightShift     19
LeftKneeMovesInward               3
LeftHeelRises                     2
Name: count, dtype: int64

## Merge features + build X/y (+ drop NaNs)

In [81]:
labels = scores_labeled[["ID", "WeakestLink"]]
data = big.merge(labels, on="ID", how="inner")

angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
y = data["WeakestLink"].astype(str)

mask = X.notna().all(axis=1)
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("X shape:", X.shape)
print("Min class count:", y.value_counts().min())


X shape: (1718, 38)
Min class count: 2


### Split training and test data

In [82]:
# Save a fixed split by ID to prevent leakage across notebooks
ids = data.loc[mask, "ID"].reset_index(drop=True)  # same rows as X/y after NaN mask

train_ids, test_ids = train_test_split(
    ids,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=region  # stratify by region target
)

joblib.dump(
    {
        "train_ids": list(train_ids),
        "test_ids": list(test_ids),
        "random_seed": RANDOM_SEED,
        "test_size": 0.2,
        "drop_ties": DROP_TIES,
        "drop_classes": DROP_CLASSES,
        "drop_weak_links": sorted(list(DROP_WEAK_LINKS)),
    },
    "split_ids.pkl"
)
print("Saved split_ids.pkl")


Saved split_ids.pkl


## Define regions

In [83]:
UPPER = {
    "ForwardHead",
    "LeftArmFallForward",
    "RightArmFallForward",
    "LeftShoulderElevation",
    "RightShoulderElevation",
}

LOWER = {
    "LeftKneeMovesInward",
    "RightKneeMovesInward",
    "LeftKneeMovesOutward",
    "RightKneeMovesOutward",
    "LeftHeelRises",
    "RightHeelRises",
}

OTHER = {
    "ExcessiveForwardLean",
    "LeftAsymmetricalWeightShift",
    "RightAsymmetricalWeightShift",
}

def to_region(lbl: str) -> str:
    if lbl in UPPER: return "Upper"
    if lbl in LOWER: return "Lower"
    if lbl in OTHER: return "Other"
    return "Other"   # safe fallback

region = y.map(to_region)

print(region.value_counts())


WeakestLink
Upper    1163
Lower     371
Other     184
Name: count, dtype: int64


## Train region classifier (CV + fit final + save)

In [84]:
region_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1
    ))
])

train_mask = ids.isin(set(train_ids))
X_train = X.loc[train_mask].reset_index(drop=True)
region_train = region.loc[train_mask].reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

rows = []
for fold, (tr, te) in enumerate(skf.split(X, region), start=1):
    Xtr, Xte = X.iloc[tr], X.iloc[te]
    rtr, rte = region.iloc[tr], region.iloc[te]

    m = region_model
    m.fit(Xtr, rtr)
    pred = m.predict(Xte)

    rows.append({
        "fold": fold,
        "accuracy": accuracy_score(rte, pred),
        "balanced_accuracy": balanced_accuracy_score(rte, pred),
        "macro_f1": f1_score(rte, pred, average="macro"),
        "weighted_f1": f1_score(rte, pred, average="weighted"),
    })

cv_df = pd.DataFrame(rows)
display(cv_df)
print("CV mean:")
display(cv_df.mean(numeric_only=True))

# Fit final on train data only + save
region_model.fit(X_train, region_train)

joblib.dump(
    {"model": region_model, "feature_cols": feature_cols},
    "region_model.pkl"
)
print("Saved: region_model.pkl")


,fold,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,0.872093,0.758825,0.801696,0.866460
1,2,0.872093,0.786278,0.807238,0.870134
2,3,0.840116,0.738584,0.759090,0.837562
3,4,0.877551,0.769987,0.811706,0.872782
4,5,0.880466,0.790502,0.816737,0.878104


CV mean:


fold                 3.000000
accuracy             0.868464
balanced_accuracy    0.768835
macro_f1             0.799293
weighted_f1          0.865008
dtype: float64

Saved: region_model.pkl
